# Pando R tutorial: human brain organoid GLI3

This notebook follows the original `new_grn/tutorials` steps 00–09. It inspects saved intermediate data, reconstructs the network from per-target fitted models, and writes regenerated outputs to `results/generated/pando_export/`. It does **not** rerun the expensive preprocessing and all 3,846 model fits. For a selected-target refit see the end of Step 07. The historical full-run scripts are retained in `src/legacy/pando_original/` for provenance; many need path and environment adaptation.


In [1]:
root <- normalizePath(getwd(), mustWork=TRUE)
while (!dir.exists(file.path(root, "dataset")) || !dir.exists(file.path(root, "results"))) {
  parent <- dirname(root)
  if (parent == root) stop("Start Jupyter from within the repository")
  root <- parent
}
Sys.setenv(GRNM_ROOT=root)
pando <- file.path(root, "dataset", "pando", "intermediates")
result <- file.path(root, "results", "pando")
suppressPackageStartupMessages(library(data.table))
cat("Repository:", root, "\nR:", R.version.string, "\n")


Repository: . 
R: R version 4.2.3 (2023-03-15) 


## 00 · Input identity
RNA and ATAC were measured in separate cells. Small metadata records are included; large original matrices are excluded and can be downloaded separately as described in the README.


In [2]:
cells <- readRDS(file.path(pando, "audit", "cell_names.rds"))
cat("ATAC cells:",length(cells[[1]]),"RNA cells:",length(cells[[2]]),"\n")
stopifnot(length(cells[[1]])==38957, length(cells[[2]])==50000)
cat("Large raw RNA/ATAC objects are separately downloaded; see README.\n")


ATAC cells: 38957 RNA cells: 50000 
Large raw RNA/ATAC objects are separately downloaded; see README.


## 01 · Genomic resources
The historical analysis uses hg38 candidate regions, motif-to-TF mappings and EnsDb v86 annotations.


In [3]:
features <- readRDS(file.path(pando,"resources","grn_features.rds"))
stopifnot(length(features)==4342)
cat("GRN features:",length(features),"\n")
stopifnot(all(file.exists(file.path(pando,"resources",c("motifs.rds","motif2tf.rds","paper_candidate_prior_regions.hg38.rds","EnsDb.Hsapiens.v86.genes.hg38.rds")))))


GRN features: 4342 


## 02 · Fragments and gene activity
Saved GeneActivity checkpoints derive from ATAC fragments; expression for modeling comes from measured RNA. This inspection does not recompute the matrix.


In [4]:
activity_qc <- fread(file.path(pando,"inputs","GeneActivity_QC.csv"))
print(activity_qc[,.(batch,cells,genes,min_gene_activity)])
stopifnot(sum(activity_qc$cells)==38957, all(activity_qc$genes==19607),all(activity_qc$min_gene_activity>0))
if (all(file.exists(file.path(pando,"fragments",activity_qc$batch,"fragments.tsv.gz")))) cat("Fragment files present.\n") else cat("Large fragments are not included; see README.\n")


              batch cells genes min_gene_activity
             <char> <int> <int>             <int>
 1:   ATAC_DAY11_SJ  4675 19607              7945
 2:   ATAC_DAY16_SJ  4987 19607              7397
 3:   ATAC_DAY21_SJ  6054 19607              7434
 4:   ATAC_DAY26_SJ  4910 19607              7358
 5:   ATAC_DAY31_SJ  5919 19607              7219
 6:    ATAC_DAY4_SJ   519 19607             13728
 7: ATAC_DAY61_2_SJ  2537 19607              7591
 8:   ATAC_DAY61_SJ  2899 19607              7261
 9:    ATAC_DAY7_SJ  1999 19607              8501
10:    ATAC_DAY9_SJ  4458 19607              8189
Large fragments are not included; see README.


## 03 · CCA and cross-modality matching
Pairs are grouped by age and cell line; repeated cell identities do not represent independent samples.


In [5]:
pairs <- fread(file.path(pando,"matches","all_pairs.tsv"))
groups <- readRDS(file.path(pando,"cca","group_summary.rds"))
cat("Matched pairs:",nrow(pairs),"groups:",length(groups),"\n")
stopifnot(nrow(pairs)==47731,length(groups)==36,all(pairs$ATAC %in% cells[[1]]),all(pairs$RNA %in% cells[[2]]))


Matched pairs: 47731 groups: 36 


## 04 · Reference classifier for QC
The saved rank-based multinomial LASSO uses WGCNAcluster instead of the original MainCellType label.


In [6]:
classifier <- readRDS(file.path(pando,"qc","nowakowski_rank_lasso.rds"))
cat("Reference cells:",length(classifier$foldid),"genes:",length(classifier$genes),"label:",classifier$label_field,"\n")
stopifnot(length(classifier$foldid)==4261,length(classifier$genes)==20477,classifier$label_field=="WGCNAcluster")


Reference cells: 4261 genes: 20477 label: WGCNAcluster 


## 05 · Metacells and pseudocells
RNA and ATAC are aggregated after matching. Check saved membership and accessibility cluster tables.


In [7]:
meta <- fread(file.path(pando,"coarse","metacell_QC.csv"))
member <- fread(file.path(pando,"coarse","pseudocell_membership.tsv"))
clusters <- fread(file.path(pando,"coarse","accessibility_clusters.csv"))
cat("Metacells:",nrow(meta),"retained:",sum(!meta$glycolysis_excluded),"pseudocells:",uniqueN(member$pseudocell),"accessibility clusters:",uniqueN(clusters$accessibility_cluster),"\n")
stopifnot(nrow(meta)==27427,sum(!meta$glycolysis_excluded)==22006,uniqueN(member$pseudocell)==6612,uniqueN(clusters$accessibility_cluster)==676)


Metacells: 27427 retained: 22006 pseudocells: 6612 accessibility clusters: 676 


## 06 · Motif mapping
The motif threshold used in scanning is distinct from the subsequent GRN FDR.


In [8]:
motif_file <- file.path(pando,"motifs","paper_motifs.rds")
if (file.exists(motif_file)) {
 suppressPackageStartupMessages({library(GenomicRanges);library(Matrix)})
 motif <- readRDS(motif_file)
 cat("Scanned regions:",nrow(motif$motifs),"motifs:",ncol(motif$motifs),"nonzero motif hits:",length(motif$motifs@x),"\n")
 stopifnot(nrow(motif$motifs)==245710,ncol(motif$motifs)==1122,length(motif$motifs@x)==7600744)
 rm(motif);gc()
} else {
 summary <- fread(file.path(pando,"motifs","motif_summary.csv"))
 print(summary)
 cat("Saved motif summary shown; large motif matrix not included, so it was not rechecked.\n")
}


   regions motifs nonzero_hits
     <int>  <int>        <int>
1:  245710   1122      7600744
Saved motif summary shown; large motif matrix not included, so it was not rechecked.


## 07 · Per-target GLM fits
Saved models implement Gaussian GLMs using TF expression × peak accessibility interactions. A selected target can be refit separately with `GRN_FIT_TARGETS=HES5 Rscript src/pando/refit_selected_targets.R`; the full set is computationally expensive.


In [9]:
fit_files <- list.files(file.path(pando,"fits"),pattern="^[0-9]+_.*\\.rds$",full.names=TRUE)
status <- fread(file.path(result,"target_status.csv"))
cat("Fit checkpoints:",length(fit_files),"fitted:",sum(status$status=="fit"),"\n")
stopifnot(length(fit_files)==3846,sum(status$status=="fit")==3446,!any(status$status=="error"))
example <- readRDS(fit_files[1]); print(example$qc)


Fit checkpoints: 3846 fitted: 3446 
  target nobs nvariables rank residual_df aliased      rsq converged
1  NEAT1 6612        664  401        6211     264 0.336327      TRUE
  inference_valid
1            TRUE


## 08 · BH correction and TF–gene network
Reconstruct all coefficients from the saved target fits, apply BH across all valid coefficient tests, aggregate significant regions, to produce the TF–gene network.


In [10]:
source(file.path(root,"src/pando/export_saved_fits.R"),local=new.env(parent=globalenv()))
original <- fread(file.path(root,"results/generated/pando_export/TF_gene_network.csv"))
cat("Exported TF–gene network:",nrow(original),"edges\n")


$attempted_targets
[1] 3846

$fitted_targets
[1] 3446

$invalid_inference_models
[1] 0

$coefficient_tests
[1] 1141828

$significant_region_edges
[1] 62354

$TF_gene_edges
[1] 49764

$TFs
[1] 738

$target_genes
[1] 3383

$gene_modules
[1] 1392

$method
[1] "Original Gaussian identity GLM; unscaled TF-expression x accessibility interactions"

$FDR
[1] "BH across all estimable coefficients; threshold 0.05"

$TF_gene_pvalues
[1] "pval_min and padj_min summarize significant region-level tests, not joint TF-gene tests"

$paper_features
[1] 4342

$features_without_v86_annotation
[1] 496

$models_with_aliased_terms
[1] 3125

$regulators_without_v86_target_names
[1] 4

$upstream_QC_label_field
[1] "WGCNAcluster"

$method_deviation
[1] "User-approved public WGCNAcluster labels replace unavailable author MainCellType labels in upstream rank-expression multinomial LASSO QC."

EXPORT_COMPLETE
Exported TF–gene network: 49764 edges


## 09 · Inspect GLI3 result
Coefficients describe conditional associations; they do not independently prove regulation or direct binding.


In [11]:
gli3 <- original[tf=="GLI3"][order(padj_min,-abs(estimate),target)]
print(gli3[,.(tf,target,estimate,padj_min)][1:min(10,.N)])
cat("GLI3 edges:",nrow(gli3),"\nR_TUTORIAL_COMPLETE\n")


        tf  target    estimate     padj_min
    <char>  <char>       <num>        <num>
 1:   GLI3   COX7B -0.36540705 1.300275e-17
 2:   GLI3  TMSB10 -0.24273510 4.157667e-17
 3:   GLI3  PGRMC1 -0.17167357 2.069522e-11
 4:   GLI3 TSPAN10  0.03627854 5.767119e-11
 5:   GLI3   EPHA7  0.08602978 1.004851e-10
 6:   GLI3  POU5F1 -0.31082808 9.542947e-09
 7:   GLI3    NONO -0.19673383 9.607186e-09
 8:   GLI3    PAX6  0.10931610 7.614250e-08
 9:   GLI3 FAM210B  0.12154601 1.850815e-07
10:   GLI3    SCG3 -0.10649416 2.132737e-07
GLI3 edges: 134 
R_TUTORIAL_COMPLETE
